## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
start_date = '20240601'
end_date = '20240630'

In [5]:
## base 

base_customer_query = f"""

    SELECT  
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        COUNT(DISTINCT user_id) active_customers
    FROM 
        canonical.clevertap_customer_order_activity 
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        -- AND serviceable = 'true'
        -- AND order_activity_source IN ('registration','login','appOpen') --inOrder
"""

df_base_customer = pd.read_sql(base_customer_query, connection)
df_base_customer

,date_range,active_customers
0,20240601 - 20240630,24560205


24.5M Active Platform Customers Base

### Does the user work/is the user a student

In [13]:
## use-case 

customer_use_case_query = f"""

    SELECT
        '{start_date}' || ' - ' || '{end_date}' AS date_range,
        usecase,
        COUNT(DISTINCT customer_id) customers
    FROM 
        (    
            (
            SELECT  
                customer_id,
                pickup_usecase.usecase
            FROM
                orders.order_logs_snapshot ols 
            JOIN 
                experiments_internal.combined_usecase_hex_10_level pickup_usecase
                ON pickup_location_hex_10 = pickup_usecase.hex_10
                AND usecase IN ('educational' , 'office')
                
            WHERE 
                yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
            )
        UNION
            (
            SELECT  
                customer_id,
                drop_usecase.usecase 
            FROM
                orders.order_logs_snapshot ols 
            JOIN 
                experiments_internal.combined_usecase_hex_10_level drop_usecase
                ON drop_location_hex_10 = drop_usecase.hex_10
                AND usecase IN ('educational' , 'office')
                
            WHERE 
                yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
            )
        UNION
            (
            SELECT 
                identity customer_id,
                'office' usecase
            FROM 
                canonical.clevertap_customer_fare_estimate
            WHERE 
                yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
                AND (pickup_address IS NOT NULL OR drop_address IS NOT NULL)
                AND 
                    (
                    regexp_like(lower(pickup_address), 'office|corp|ltd|llc|inc|headquarter|tech|solutions|innovations|systems|software|enterprises|business|labs|digital|network')
                    OR
                    regexp_like(lower(drop_address), 'office|corp|ltd|llc|inc|headquarter|tech|solutions|innovations|systems|software|enterprises|business|labs|digital|network')
                    )
            GROUP BY 1
            )
        UNION 
            (
            SELECT 
                identity customer_id,
                'educational' usecase
            FROM 
                canonical.clevertap_customer_fare_estimate
            WHERE 
                yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
                AND (pickup_address IS NOT NULL OR drop_address IS NOT NULL)
                AND 
                    (
                    regexp_like(lower(pickup_address), 'school|university|college|institute|academy|elementary|primary|secondary|kindergarten') --campus
                    OR
                    regexp_like(lower(drop_address), 'school|university|college|institute|academy|elementary|primary|secondary|kindergarten') --campus
                    )
            GROUP BY 1
            )
        )
    GROUP BY 1,2

"""

df_customer_use_case = pd.read_sql(customer_use_case_query, connection)
df_customer_use_case

,date_range,usecase,customers
0,20240601 - 20240630,educational,6021176
1,20240601 - 20240630,office,4991120


In [18]:
df_use_case_merge = pd.merge(df_base_customer, df_customer_use_case, how='left', on='date_range')
df_use_case_merge['% distribution'] = round(df_use_case_merge['customers']*100.00/df_use_case_merge['active_customers'], 2)
df_use_case_merge

,date_range,active_customers,usecase,customers,% distribution
0,20240601 - 20240630,24560205,educational,6021176,24.52
1,20240601 - 20240630,24560205,office,4991120,20.32


In [ ]:
### 